# EAHN — HiDF Training + Cross-Dataset Evaluation (Phase 18)

Single-notebook workflow:
1. Train EAHN on **HiDF** (balanced, 50/50, KDD 2025) — primary experiment.
2. Auto-eval on **HiDF test split** (in-distribution).
3. Auto-eval on **FF++ per manipulation** (5 cells × 100R+100F).
4. Auto-eval on **Celeb-DF v2** (1 cell × 100R+100F balanced).
5. Run explanation suite on HiDF test set.
6. Aggregate everything, build two zips (complete + analysis-essentials).

Run every cell **top to bottom**. No inline patches — all logic is in the repo at `github.com/umardrazbhatti/EahnHiDF`.

| Cell | Purpose | Gate before next |
|------|---------|-----------------|
| 1 | Configuration + path detection | All three dataset roots resolve |
| 2 | File-count sanity (HiDF, FF++, Celeb-DF) | All counts as expected |
| 3 | Clean previous run | — |
| 4 | Clone EahnHiDF repo | Phase 18.x commits visible |
| 5 | Install deps + import chain | ALL IMPORTS OK |
| 6 | Run `scripts/HiDF_verify_dataset.py --dataset_name hidf` | Exit 0 |
| 6b | HiDF loader sanity (both classes in batch) | VERIFICATION PASSED |
| 6c | (Optional) restore from uploaded checkpoint | Skip Cell 7 if Restore complete |
| 7 | Train + auto-eval (HiDF + FF++ × 5 + Celeb-DF + explanation) | Val AUC > 0.55 at epoch 1, > 0.75 at epoch 5 |
| 7c | Collapse diagnostic | No collapse detected |
| 8 | Metrics tables (HiDF / FF++ × 5 / Celeb-DF / explanation) | — |
| 9 | Detection graphs grid | — |
| 10 | Cross-dataset summary chart | — |
| 11 | Heatmap overlays | — |
| 12 | Build analysis-essentials bundle | — |
| 13 | Zip × 2: complete + analysis-essentials | Both files in `/kaggle/working/` |

**Acceptance gates** (from `EahnHiDF_Project_Bible.md` §7):
- Gate 4: Val AUC > 0.55 at epoch 1 → if fails, STOP and debug.
- Gate 5: Val AUC > 0.75 at epoch 5 → if fails, architecture/loss is broken.
- Gate 8: Inter-sample cosine mean < 0.70 → otherwise explanation collapse.


---
## Cell 1 — Configuration + path detection

Edit `EPOCHS` only if you want a different training length. Default 20 with early-stop patience 5.


In [ ]:
import os, glob

# ════════════════════════════════════════════════════════════════════════════
# ║                       NOTEBOOK CONFIGURATION                              ║
# ════════════════════════════════════════════════════════════════════════════

EPOCHS     = 20
BATCH_SIZE = 4

# Dataset roots — Kaggle mounts. Auto-detected if missing.
HIDF_DATA_ROOT    = "/kaggle/input/datasets/rahamimran/hidf-real-fake-dataset/Fake Real Dataset"
FFPP_DATA_ROOT    = "/kaggle/input/datasets/umardrazbhatti/ffpp-c23-custom-layout/ffpp_data"
CELEBDF_DATA_ROOT = "/kaggle/input/datasets/reubensuju/celeb-df-v2"

REPO_URL   = "https://github.com/umardrazbhatti/EahnHiDF.git"
REPO_DIR   = "/kaggle/working/EahnHiDF"
OUTPUT_DIR = "/kaggle/working/outputs"
CACHE_DIR  = "/kaggle/working/.face_cache"

# ════════════════════════════════════════════════════════════════════════════

def _find_under(base, predicate, max_depth=6):
    """Walk /kaggle/input looking for a directory matching predicate(root)."""
    for root, dirs, _ in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        if predicate(root):
            return root
    return None

# HiDF auto-detect
if not os.path.isdir(HIDF_DATA_ROOT):
    print(f"[WARN] HIDF_DATA_ROOT not found: {HIDF_DATA_ROOT}")
    print(f"       Attempting auto-detect ...")
    fallback = _find_under(
        "/kaggle/input",
        lambda r: os.path.isdir(os.path.join(r, "Real-vid")) and
                  os.path.isdir(os.path.join(r, "Fake-vid"))
    )
    if fallback:
        HIDF_DATA_ROOT = fallback
        print(f"       Found: {HIDF_DATA_ROOT}")
    else:
        print("[ERROR] HiDF root not found. Available inputs:")
        for e in sorted(glob.glob("/kaggle/input/*/*"))[:30]:
            print(f"   {e}")
        raise SystemExit("Attach HiDF dataset and fix HIDF_DATA_ROOT in Cell 1.")

# FF++ auto-detect
if not os.path.isdir(FFPP_DATA_ROOT):
    print(f"[WARN] FFPP_DATA_ROOT not found: {FFPP_DATA_ROOT}")
    fallback = _find_under(
        "/kaggle/input",
        lambda r: os.path.isdir(os.path.join(r, "original_sequences")) and
                  os.path.isdir(os.path.join(r, "manipulated_sequences"))
    )
    if fallback:
        FFPP_DATA_ROOT = fallback
        print(f"       Found: {FFPP_DATA_ROOT}")
    else:
        print("[WARN] FF++ root not found. FF++ cross-eval will be SKIPPED.")
        FFPP_DATA_ROOT = ""

# Celeb-DF auto-detect
if not os.path.isdir(CELEBDF_DATA_ROOT):
    print(f"[WARN] CELEBDF_DATA_ROOT not found: {CELEBDF_DATA_ROOT}")
    fallback = _find_under(
        "/kaggle/input",
        lambda r: os.path.isdir(os.path.join(r, "Celeb-real")) and
                  os.path.isdir(os.path.join(r, "Celeb-synthesis"))
    )
    if fallback:
        CELEBDF_DATA_ROOT = fallback
        print(f"       Found: {CELEBDF_DATA_ROOT}")
    else:
        print("[WARN] Celeb-DF root not found. Celeb-DF cross-eval will be SKIPPED.")
        CELEBDF_DATA_ROOT = ""

print("=" * 70)
print(f"HIDF_DATA_ROOT      : {HIDF_DATA_ROOT}")
print(f"FFPP_DATA_ROOT      : {FFPP_DATA_ROOT or '(missing — FF++ cross-eval disabled)'}")
print(f"CELEBDF_DATA_ROOT   : {CELEBDF_DATA_ROOT or '(missing — Celeb-DF cross-eval disabled)'}")
print(f"REPO_DIR            : {REPO_DIR}")
print(f"OUTPUT_DIR          : {OUTPUT_DIR}")
print(f"EPOCHS              : {EPOCHS}")
print(f"BATCH_SIZE          : {BATCH_SIZE}")
print("=" * 70)
print("This run: train on HiDF, eval on HiDF in-dist + FF++ × 5 + Celeb-DF + explanation")


---
## Cell 2 — File-count sanity check
Expected counts:

- HiDF: **4,361 real + 4,361 fake**
- FF++ c23: **1,000 real + 5 × 1,000 fake**
- Celeb-DF v2: **890 real + 5,639 fake** with test list ~518 entries


In [ ]:
import glob, os

errors = []

# ── HiDF check ───────────────────────────────────────────────────────────────
print("HiDF:")
hidf_real = glob.glob(os.path.join(HIDF_DATA_ROOT, "Real-vid", "*.mp4"))
hidf_fake = glob.glob(os.path.join(HIDF_DATA_ROOT, "Fake-vid", "*.mp4"))
status_r = "OK" if len(hidf_real) > 4000 else "FAIL"
status_f = "OK" if len(hidf_fake) > 4000 else "FAIL"
print(f"  [{status_r:^4}]  Real-vid  {len(hidf_real):>6} .mp4  (expected 4361)")
print(f"  [{status_f:^4}]  Fake-vid  {len(hidf_fake):>6} .mp4  (expected 4361)")
if status_r == "FAIL" or status_f == "FAIL":
    errors.append("HiDF counts off — re-verify HIDF_DATA_ROOT")

# ── FF++ check ───────────────────────────────────────────────────────────────
if FFPP_DATA_ROOT:
    print("\nFF++ c23:")
    real_dir = os.path.join(FFPP_DATA_ROOT, "original_sequences", "youtube", "c23", "videos")
    n_real = len(glob.glob(os.path.join(real_dir, "*.mp4")))
    s = "OK" if n_real >= 1000 else "FAIL"
    print(f"  [{s:^4}]  original (real)        {n_real:>6} .mp4  (expected 1000)")
    if s == "FAIL":
        errors.append("FF++ real count off")
    METHODS = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]
    for m in METHODS:
        d = os.path.join(FFPP_DATA_ROOT, "manipulated_sequences", m, "c23", "videos")
        c = len(glob.glob(os.path.join(d, "*.mp4")))
        s = "OK" if c >= 1000 else "FAIL"
        print(f"  [{s:^4}]  {m:<22} {c:>6} .mp4  (expected 1000)")
        if s == "FAIL":
            errors.append(f"FF++ {m} count off")

# ── Celeb-DF v2 check ────────────────────────────────────────────────────────
if CELEBDF_DATA_ROOT:
    print("\nCeleb-DF v2:")
    for label, sub, exp in [
        ("Celeb-real", "Celeb-real", 590),
        ("YouTube-real", "YouTube-real", 300),
        ("Celeb-synthesis", "Celeb-synthesis", 5639),
    ]:
        d = os.path.join(CELEBDF_DATA_ROOT, sub)
        n = len(glob.glob(os.path.join(d, "*.mp4")))
        s = "OK" if n > 0 else "FAIL"
        print(f"  [{s:^4}]  {label:<22} {n:>6} .mp4  (expected {exp})")
        if s == "FAIL":
            errors.append(f"Celeb-DF {label} missing")

    test_list = os.path.join(CELEBDF_DATA_ROOT, "List_of_testing_videos.txt")
    if os.path.isfile(test_list):
        with open(test_list) as f:
            n_entries = sum(1 for line in f if line.strip())
        s = "OK" if 400 < n_entries < 600 else "WARN"
        print(f"  [{s:^4}]  test list entries:      {n_entries}  (expected ~518)")
    else:
        print(f"  [FAIL]  List_of_testing_videos.txt not found")
        errors.append("Celeb-DF test list missing")

# ── Summary ──────────────────────────────────────────────────────────────────
print()
if errors:
    print("ERRORS:")
    for e in errors:
        print(f"  - {e}")
    raise SystemExit("Fix dataset paths before continuing.")
else:
    print("All dataset checks passed.  Proceed to Cell 3.")


---
## Cell 3 — Clean previous run
Deletes the previous clone and outputs. Preserves face cache (saves ~3–5 min on re-runs).
Dataset under `/kaggle/input/` is never touched.


In [ ]:
import shutil, os

to_remove = [
    REPO_DIR,
    OUTPUT_DIR,
    "/kaggle/working/eahn_hidf_complete.zip",
    "/kaggle/working/eahn_hidf_analysis.zip",
]
for path in to_remove:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Removed dir  : {path}")
    elif os.path.isfile(path):
        os.remove(path)
        print(f"Removed file : {path}")
    else:
        print(f"Not present  : {path}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.isdir(CACHE_DIR):
    n_cached = sum(len(files) for _, _, files in os.walk(CACHE_DIR))
    print(f"[face_cache] preserved: {n_cached} cached files at {CACHE_DIR}")
else:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f"[face_cache] empty cache created at {CACHE_DIR} (first run will be slower)")

print("\nClean. Proceed to Cell 4.")


---
## Cell 4 — Clone the EahnHiDF repo
⚠️ **Internet must be ON**: Kaggle right panel → Settings → Internet → On.
Expected: commits up to **Phase 18.5** visible in `git log`.


In [ ]:
import os, sys, subprocess

ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")

if ret != 0 or not os.path.isdir(REPO_DIR):
    raise SystemExit(
        "Clone failed.\n"
        "Fix: Enable Internet in Kaggle Settings (right panel → Internet → On).\n"
        "Then re-run this cell."
    )

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_DIR).decode().strip()
log    = subprocess.check_output(["git", "log", "--oneline", "-10"], cwd=REPO_DIR).decode().strip()
print(f"\nClone successful.")
print(f"  Branch : {branch}")
print(f"  Recent commits:")
for line in log.splitlines():
    print(f"    {line}")
print("\nRepo contents:")
for f in sorted(os.listdir(REPO_DIR)):
    if not f.startswith("."):
        print(f"  {f}")


---
## Cell 5 — Install dependencies + import chain check
Must print **ALL IMPORTS OK** before continuing.


In [ ]:
import subprocess, sys, importlib, os

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Flush stale module cache
stale = [k for k in list(sys.modules) if any(k.startswith(p) for p in
         ["config", "data", "models", "losses", "xai", "metrics", "utils", "scripts"])]
for k in stale:
    del sys.modules[k]

print("Installing requirements...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     f"{REPO_DIR}/requirements.txt"],
    check=True
)
print("Installation complete.\n")

pkg_map = {"torch":"torch","torchvision":"torchvision","timm":"timm",
           "sklearn":"sklearn","cv2":"cv2","PIL":"PIL","tqdm":"tqdm"}
for name, mod_name in pkg_map.items():
    try:
        m = importlib.import_module(mod_name)
        print(f"  OK      {name:<15} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"  MISSING {name}")

print("\nChecking full import chain...")
try:
    from HiDF_config import EAHNConfig                            # noqa
    from data.datasets import DeepfakeDataset                # noqa
    from data.celebdf_dataset import CelebDFv2TestDataset    # noqa
    from data.collate import deepfake_collate_fn             # noqa
    from data.transforms import get_transforms               # noqa
    from data.face_align import FaceAligner                  # noqa
    from models.eahn import EAHN                             # noqa
    from losses.classification import ClassificationLoss     # noqa
    from losses.explanation import ExplanationLoss           # noqa
    from losses.temporal import TemporalConsistencyLoss      # noqa
    from metrics.detection import DetectionMetrics           # noqa
    from scripts.train_real import main as train_main        # noqa
    from scripts.evaluate import run_evaluation              # noqa
    from scripts.dashboard import show_dashboard             # noqa

    # Verify HiDF support is in the cloned repo
    import inspect
    from data.datasets import DeepfakeDataset
    if not hasattr(DeepfakeDataset, "_build_hidf"):
        raise ImportError("DeepfakeDataset._build_hidf not found — repo is on a pre-Phase-18 commit.")

    # Verify config supports hidf
    cfg = EAHNConfig()
    if not hasattr(cfg, "hidf_root"):
        raise ImportError("EAHNConfig.hidf_root not found — Phase 18.2 not applied.")
    if not hasattr(cfg, "ffpp_cross_eval"):
        raise ImportError("EAHNConfig.ffpp_cross_eval not found — Phase 18.2 not applied.")

    print("\nALL IMPORTS OK — Phase 18 amendments confirmed. Proceed to Cell 6.")
except ImportError as e:
    raise SystemExit(
        f"\nIMPORT FAILED: {e}\n"
        "The Phase 18 amendments are missing or the clone in Cell 4 pulled an old commit.\n"
        "Action: re-run from Cell 3 to get a fresh clone."
    )


---
## Cell 6 — Run `scripts/HiDF_verify_dataset.py` (HiDF mode)
This invokes the Phase 18.5 verifier. Must exit 0.
Confirms HiDF Real-vid/Fake-vid counts, FF++ cross-eval paths, Celeb-DF cross-eval paths.


In [ ]:
import os, sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

cmd_parts = [
    f"python scripts/HiDF_verify_dataset.py",
    f"--dataset_name hidf",
    f'--hidf_root "{HIDF_DATA_ROOT}"',
    f'--output_dir "{OUTPUT_DIR}"',
    f'--cache_dir "{CACHE_DIR}"',
]
if FFPP_DATA_ROOT:
    cmd_parts.append(f"--ffpp_cross_eval")
    cmd_parts.append(f'--ffpp_cross_root "{FFPP_DATA_ROOT}"')
if CELEBDF_DATA_ROOT:
    cmd_parts.append(f"--celebdf_eval")
    cmd_parts.append(f'--celebdf_root "{CELEBDF_DATA_ROOT}"')

cmd = " ".join(cmd_parts)
print("Running:")
print(f"  {cmd}\n")
ret = os.system(cmd)
if ret != 0:
    print(f"\n[ERROR] verify_dataset.py exited {ret}. Inspect the output above.")
else:
    print("\nverify_dataset.py: OK. Proceed to Cell 6b.")


---
## Cell 6b — HiDF loader sanity check
Builds the HiDF train dataset (source-grouped 80/10/10 split), pulls 3 batches with shuffle.
Asserts both classes are present, balance is reasonable, and source IDs don't leak across splits.


In [ ]:
import os, sys, torch, re
from collections import Counter
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

stale = [k for k in list(sys.modules) if any(k.startswith(p) for p in
         ["data","models","losses","xai","metrics","utils","scripts","config"])]
for k in stale: del sys.modules[k]

from HiDF_config import EAHNConfig
from data.datasets import DeepfakeDataset
from data.collate  import deepfake_collate_fn
from torch.utils.data import DataLoader

# Build HiDF training config
_cfg = EAHNConfig(
    hidf_root      = HIDF_DATA_ROOT,
    output_dir     = OUTPUT_DIR,
    cache_dir      = CACHE_DIR,
    dataset_name   = "hidf",
    num_frames     = 4,        # small T for fast verification
    frame_size     = 224,
    train_split    = 0.8,
    val_split      = 0.1,
    device         = "auto",
    num_workers    = 0,
    max_per_class  = 0,        # no cap for verification
)

print("Building HiDF train dataset...")
train_ds = DeepfakeDataset(_cfg, mode="train", dataset_type="hidf")
val_ds   = DeepfakeDataset(_cfg, mode="val",   dataset_type="hidf")
test_ds  = DeepfakeDataset(_cfg, mode="test",  dataset_type="hidf")

# ── Per-split counts ─────────────────────────────────────────────────────────
SRC_RE = re.compile(r"^(c\d+)")
def src_id(p):
    stem = os.path.splitext(os.path.basename(p))[0]
    m = SRC_RE.match(stem)
    return m.group(1) if m else stem

def split_stats(name, ds):
    samples = ds.samples
    paths   = [s["video_path"] if isinstance(s, dict) else s[0] for s in samples]
    labels  = [s["label"]      if isinstance(s, dict) else s[1] for s in samples]
    n_real  = labels.count(0)
    n_fake  = labels.count(1)
    sources = set(src_id(p) for p in paths)
    print(f"  {name:<6}  total={len(samples):>6}  real={n_real:>5}  fake={n_fake:>5}  unique_sources={len(sources):>5}")
    return sources

print("\nPer-split sample counts:")
src_train = split_stats("train", train_ds)
src_val   = split_stats("val",   val_ds)
src_test  = split_stats("test",  test_ds)

# ── Source-leakage check ─────────────────────────────────────────────────────
print("\nSource-ID leakage check (must all be 0):")
print(f"  train ∩ val   = {len(src_train & src_val)}")
print(f"  train ∩ test  = {len(src_train & src_test)}")
print(f"  val   ∩ test  = {len(src_val   & src_test)}")

assert len(src_train & src_val)  == 0, "LEAKAGE: train and val share source IDs"
assert len(src_train & src_test) == 0, "LEAKAGE: train and test share source IDs"
assert len(src_val   & src_test) == 0, "LEAKAGE: val and test share source IDs"

# ── Batch check ──────────────────────────────────────────────────────────────
loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                    collate_fn=deepfake_collate_fn, num_workers=0)

print("\nInspecting 3 batches (shuffle=True, batch_size=8):")
saw_real = saw_fake = False
for i, b in enumerate(loader):
    u = b["label"].unique().tolist()
    r = int((b["label"] == 0).sum()); f = int((b["label"] == 1).sum())
    print(f"  batch {i}: real={r} fake={f} unique={u} frames_shape={tuple(b['frames'].shape)}")
    if r > 0: saw_real = True
    if f > 0: saw_fake = True
    if i == 2: break

assert saw_real and saw_fake, "Loader not delivering both classes across 3 batches."

print()
print("=" * 70)
print("VERIFICATION PASSED — HiDF loader is healthy, no source-ID leakage.")
print("Proceed to Cell 6c (optional restore) or Cell 7 (train).")
print("=" * 70)


---
## Cell 6c — Restore from uploaded checkpoint (optional)

If you have a previous `eahn_hidf_best.pth` uploaded as a private Kaggle Dataset, this cell copies it (and the face cache, if present) into `OUTPUT_DIR` / `CACHE_DIR`. Lets you **skip Cell 7 (training)** and re-evaluate.

If no upload is found, the cell prints instructions and exits cleanly — Cell 7 then trains from scratch.


In [ ]:
import os, shutil

def _find_uploaded_checkpoint(base="/kaggle/input", max_depth=3):
    if not os.path.isdir(base):
        return None
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        for cand in ("eahn_hidf_best.pth", "best_model.pth"):
            if cand in files:
                return root, cand
    return None

result = _find_uploaded_checkpoint()

if result is None:
    print("[INFO] No uploaded checkpoint found under /kaggle/input/.")
    print()
    print("To skip Cell 7 (training):")
    print("  1. Zip your local outputs (must include eahn_hidf_best.pth).")
    print("  2. Create a private Kaggle Dataset from that zip.")
    print("  3. Attach it: right panel -> Add Data -> Your Datasets.")
    print("  4. Re-run this cell.")
    print()
    print("Otherwise, proceed to Cell 7 to train from scratch.")
else:
    upload_dir, ckpt_name = result
    print(f"[INFO] Found uploaded checkpoint: {upload_dir}/{ckpt_name}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    src_ckpt = os.path.join(upload_dir, ckpt_name)
    dst_ckpt = os.path.join(OUTPUT_DIR, "eahn_hidf_best.pth")
    shutil.copy2(src_ckpt, dst_ckpt)
    size_mb = os.path.getsize(dst_ckpt) / (1024 * 1024)
    print(f"  Copied checkpoint ({size_mb:.1f} MB) -> {dst_ckpt}")

    src_cache = os.path.join(upload_dir, ".face_cache")
    if os.path.isdir(src_cache):
        os.makedirs(CACHE_DIR, exist_ok=True)
        n = 0
        for item in os.listdir(src_cache):
            s = os.path.join(src_cache, item)
            d = os.path.join(CACHE_DIR, item)
            if os.path.isdir(s):
                if os.path.isdir(d): shutil.rmtree(d)
                shutil.copytree(s, d)
                n += sum(len(files) for _, _, files in os.walk(d))
            else:
                shutil.copy2(s, d); n += 1
        print(f"  Copied face cache ({n} files) -> {CACHE_DIR}")

    aux = ["training_history.csv", "logs.csv", "metrics.csv", "metrics.json"]
    for a in aux:
        sa = os.path.join(upload_dir, a)
        if os.path.isfile(sa):
            shutil.copy2(sa, os.path.join(OUTPUT_DIR, a))

    print()
    print("=" * 60)
    print("Restore complete.")
    print("Cell 7 will detect the checkpoint and SKIP training.")
    print("=" * 60)


---
## Cell 7 — Train EAHN on HiDF + auto-evaluate everything

This is the main run. With a cold face cache, expect ~5–10 min per epoch on T4.
With a warm cache, ~3–5 min per epoch. 20 epochs total → roughly **2–4 hours**.

After training, evaluation runs automatically (via `--eval_after_train`):
- HiDF in-distribution test set (built-in, full split)
- FF++ per-manipulation cross-eval × 5 (`--ffpp_cross_eval`, capped 100R+100F each)
- Celeb-DF balanced cross-eval (`--celebdf_eval`, capped 100R+100F)
- Explanation suite (intrinsic, faithfulness, stability, collapse, model-randomization)

**Gates:** Val AUC > 0.55 at epoch 1, > 0.75 at epoch 5. If either fails, the run is broken.


In [ ]:
import os, sys, shutil
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

_existing_ckpt = os.path.join(OUTPUT_DIR, "eahn_hidf_best.pth")
if os.path.isfile(_existing_ckpt):
    _size_mb = os.path.getsize(_existing_ckpt) / (1024 * 1024)
    print("=" * 70)
    print(f"[SKIP] eahn_hidf_best.pth already exists ({_size_mb:.1f} MB)")
    print(f"   {_existing_ckpt}")
    print()
    print("Cell 6c likely restored a previous checkpoint.")
    print("Training is SKIPPED to avoid overwriting it.")
    print()
    print("To train from scratch:")
    print(f"   import os; os.remove({_existing_ckpt!r})")
    print("=" * 70)
else:
    # Clear face cache to avoid T-mismatch collision with Cell 6b (T=4)
    if os.path.isdir(CACHE_DIR):
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR, exist_ok=True)
        print("Face cache cleared (Cell 6b used T=4; training uses T=16).")

    cmd_parts = [
        "python HiDF_run_full_pipeline.py",
        f'--data_root          "{HIDF_DATA_ROOT}"',
        f'--hidf_root          "{HIDF_DATA_ROOT}"',
        f'--dataset_name       hidf',
        f'--output_dir         "{OUTPUT_DIR}"',
        f'--cache_dir          "{CACHE_DIR}"',
        f'--epochs             {EPOCHS}',
        f'--batch_size         {BATCH_SIZE}',
        f'--num_workers        2',
        f'--grad_accum_steps   4',
        f'--gamma              0.1',
        f'--attn_temp_init     0.0',
        f'--attn_diversity_weight 5.0',
        f'--cls_dropout_p      0.0',
        f'--cls_loss_type      bce',
        f'--lambda1            0.3',
        f'--lambda2            0.2',
        f'--alpha              0.3',
        f'--beta               0.5',
        f'--label_smoothing    0.05',
        f'--use_amp',
        f'--grad_checkpoint',
        f'--eval_after_train',
    ]
    if FFPP_DATA_ROOT:
        cmd_parts.append('--ffpp_cross_eval')
        cmd_parts.append(f'--ffpp_cross_root "{FFPP_DATA_ROOT}"')
    if CELEBDF_DATA_ROOT:
        cmd_parts.append('--celebdf_eval')
        cmd_parts.append(f'--celebdf_root "{CELEBDF_DATA_ROOT}"')

    cmd = " ".join(cmd_parts)
    print("Running training + auto-eval:")
    print(f"  {cmd}\n")

    ret = os.system(cmd)
    print()
    if ret != 0:
        print(f"[ERROR] Pipeline exited with code {ret}. Inspect log above.")
    else:
        print("Cell 7 complete.")
        print("Outputs (in OUTPUT_DIR):")
        for f in [
            "eahn_hidf_best.pth",
            "metrics.csv",
            "explanation_metrics.json",
            "eval/hidf_test_metrics.json",
            "eval/celebdf_test_metrics.json",
        ]:
            path = os.path.join(OUTPUT_DIR, f)
            print(f"  {'OK' if os.path.isfile(path) else '--'}  {f}")
        # Check FF++ cross-eval cells
        eval_root = os.path.join(OUTPUT_DIR, "eval")
        if os.path.isdir(eval_root):
            for d in sorted(os.listdir(eval_root)):
                if d.startswith("ffpp_cross_"):
                    print(f"  OK  eval/{d}/")


---
## Cell 7c — Explanation-collapse diagnostic (gate 8)
Must print **No collapse detected** to clear gate 8.


In [ ]:
import os, json, pandas as pd

# Prefer the explanation_metrics.json which has the diagnostic block
exp_path = os.path.join(OUTPUT_DIR, "explanation_metrics.json")
if not os.path.exists(exp_path):
    print(f"explanation_metrics.json not found at {exp_path}")
else:
    with open(exp_path) as f:
        exp = json.load(f)

    # Diagnostic keys may live under "collapse_diagnostics" or flat
    cd = exp.get("collapse_diagnostics", exp)
    inter   = cd.get("inter_sample_cosine_mean", cd.get("inter_sample_cosine"))
    mode    = cd.get("peak_mode_share")
    mt_std  = cd.get("m_t_std_mean", cd.get("m_t_std"))

    warnings = []
    if inter  is not None and inter  > 0.70: warnings.append(f"inter_sample_cosine_mean={inter:.3f} (>0.70)")
    if mode   is not None and mode   > 0.50: warnings.append(f"peak_mode_share={mode:.3f} (>0.50)")
    if mt_std is not None and mt_std < 0.05: warnings.append(f"m_t_std_mean={mt_std:.3f} (<0.05)")

    print("Explanation diagnostics:")
    if inter  is not None: print(f"  inter_sample_cosine_mean = {inter:.4f}  (threshold: < 0.70)")
    if mode   is not None: print(f"  peak_mode_share          = {mode:.4f}  (threshold: < 0.50)")
    if mt_std is not None: print(f"  m_t_std_mean             = {mt_std:.4f}  (threshold: > 0.05)")

    print()
    if warnings:
        print("EXPLANATION COLLAPSE DETECTED:")
        for w in warnings: print(f"   - {w}")
        print("The explanation head produced near-identical maps across samples.")
        print("This is a known carry-forward issue (parent project Ch24, SSIM ~0.89).")
        print("Document in Ch4 as a future-work item; do not retrain solely for this.")
    else:
        print("No collapse detected. Explanation suite is healthy.")


---
## Cell 8 — Metrics tables (HiDF / FF++ × 5 / Celeb-DF / explanation)


In [ ]:
import os, json
import pandas as pd
from IPython.display import display, HTML

eval_root = os.path.join(OUTPUT_DIR, "eval")

# ── Helper to load any metrics JSON ──────────────────────────────────────────
def load_json(path):
    if not os.path.isfile(path): return None
    with open(path) as f:
        return json.load(f)

def json_to_df(blob, label):
    rows = []
    for k, v in blob.items():
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            rows.append((k, round(v, 4)))
    return pd.DataFrame(rows, columns=["Metric", label])

# ── 1. HiDF in-distribution ──────────────────────────────────────────────────
display(HTML("<h2>HiDF in-distribution test</h2>"))
hidf = load_json(os.path.join(eval_root, "hidf_test_metrics.json")) \
    or load_json(os.path.join(eval_root, "metrics.json")) \
    or load_json(os.path.join(OUTPUT_DIR, "metrics.json"))
if hidf:
    display(json_to_df(hidf, "HiDF (in-dist)"))
    if "auc_roc" in hidf:
        auc = hidf["auc_roc"]
        msg = "STRONG" if auc >= 0.85 else "GOOD" if auc >= 0.70 else "WEAK" if auc >= 0.55 else "NEAR-CHANCE"
        print(f"\nHiDF AUC-ROC = {auc:.4f} → {msg}")
else:
    print("HiDF metrics not found.")

# ── 2-6. FF++ per manipulation ───────────────────────────────────────────────
display(HTML("<h2>FF++ per-manipulation cross-evaluation (100R + 100F each)</h2>"))
FFPP_METHODS = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]
ffpp_rows = []
for m in FFPP_METHODS:
    p = os.path.join(eval_root, f"ffpp_cross_{m}", "metrics.json")
    blob = load_json(p)
    if blob:
        row = {"Manipulation": m}
        for k in ("auc_roc", "auc_pr", "f1_at_optimal", "precision_at_optimal",
                  "recall_at_optimal", "balanced_accuracy_at_optimal"):
            if k in blob: row[k] = round(blob[k], 4)
        ffpp_rows.append(row)
    else:
        ffpp_rows.append({"Manipulation": m, "auc_roc": "MISSING"})

if ffpp_rows:
    ffpp_df = pd.DataFrame(ffpp_rows).set_index("Manipulation")
    display(ffpp_df)
else:
    print("No FF++ cross-eval results found.")

# ── 7. Celeb-DF ──────────────────────────────────────────────────────────────
display(HTML("<h2>Celeb-DF v2 cross-evaluation (100R + 100F balanced sample)</h2>"))
celeb = load_json(os.path.join(eval_root, "celebdf_test_metrics.json")) \
    or load_json(os.path.join(OUTPUT_DIR, "celebdf_test_metrics.json"))
if celeb:
    display(json_to_df(celeb, "Celeb-DF v2"))
    if "auc_roc" in celeb:
        auc = celeb["auc_roc"]
        msg = "Good transfer" if auc >= 0.65 else "Weak transfer (expected)" if auc >= 0.50 else "Below chance"
        print(f"\nCeleb-DF AUC-ROC = {auc:.4f} → {msg}")
else:
    print("Celeb-DF metrics not found.")

# ── 8. Explanation suite ─────────────────────────────────────────────────────
display(HTML("<h2>Explanation suite (HiDF test)</h2>"))
exp = load_json(os.path.join(OUTPUT_DIR, "explanation_metrics.json"))
if exp:
    flat = []
    for section, payload in exp.items():
        if isinstance(payload, dict):
            for k, v in payload.items():
                flat.append((f"{section}.{k}", v))
        else:
            flat.append((section, payload))
    edf = pd.DataFrame(flat, columns=["Metric", "Value"])
    edf["Value"] = edf["Value"].apply(lambda x: round(float(x), 4) if isinstance(x, (int, float)) else x)
    display(edf)
else:
    print("explanation_metrics.json not found.")


---
## Cell 9 — Detection graphs (ROC, PR, confusion matrix, score distribution)
Per evaluation cell. 7 cells × 4 plots = up to 28 graphs.


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")

def _find_plot(prefixes, leaf):
    """Search multiple plot locations for a {prefix}_{leaf} or {leaf} file."""
    candidates = []
    for prefix in prefixes:
        candidates.append(os.path.join(PLOT_DIR, f"{prefix}_{leaf}"))
    candidates.append(os.path.join(PLOT_DIR, leaf))
    candidates.append(os.path.join(OUTPUT_DIR, leaf))
    # Also check eval/ subdirectories for per-cell plots
    for prefix in prefixes:
        cell_dir = os.path.join(EVAL_DIR, prefix)
        if os.path.isdir(cell_dir):
            for fn in os.listdir(cell_dir):
                if fn.endswith(leaf):
                    candidates.append(os.path.join(cell_dir, fn))
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

def _show_row(title, prefix_list):
    leaves = [
        ("roc_curve.png",          "ROC"),
        ("pr_curve.png",           "Precision-Recall"),
        ("confusion_matrix.png",   "Confusion Matrix"),
        ("score_distribution.png", "Score Distribution"),
    ]
    found = [(p, sub) for leaf, sub in leaves for p in [_find_plot(prefix_list, leaf)] if p]
    if not found:
        print(f"[{title}] no plots found.")
        return
    n = len(found)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 4.5))
    if n == 1: axes = [axes]
    for ax, (path, sub) in zip(axes, found):
        ax.imshow(mpimg.imread(path))
        ax.set_title(sub, fontsize=11, fontweight="bold")
        ax.axis("off")
    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

# HiDF in-distribution
_show_row("HiDF in-distribution test", ["hidf", "ffpp"])  # fallback name

# FF++ per manipulation
for m in ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]:
    _show_row(f"FF++ cross-eval — {m}", [f"ffpp_cross_{m}"])

# Celeb-DF
_show_row("Celeb-DF v2 cross-eval", ["celebdf"])

# Training curves
val_curve = os.path.join(PLOT_DIR, "val_accuracy_curves.png")
loss_curve = os.path.join(PLOT_DIR, "loss_curves.png")
for path, title in [(loss_curve, "Training loss curves"),
                    (val_curve, "Validation accuracy curves")]:
    if os.path.exists(path):
        plt.figure(figsize=(9, 5))
        plt.imshow(mpimg.imread(path))
        plt.title(title, fontsize=12, fontweight="bold")
        plt.axis("off")
        plt.show()


---
## Cell 10 — Cross-dataset summary chart
Single bar chart comparing AUC-ROC across all 7 evaluation cells.


In [ ]:
import os, json
import matplotlib.pyplot as plt
import pandas as pd

eval_root = os.path.join(OUTPUT_DIR, "eval")

def load_auc(path):
    if not os.path.isfile(path): return None
    with open(path) as f:
        blob = json.load(f)
    return blob.get("auc_roc")

rows = []

# HiDF in-distribution
for p in [
    os.path.join(eval_root, "hidf_test_metrics.json"),
    os.path.join(eval_root, "metrics.json"),
    os.path.join(OUTPUT_DIR, "metrics.json"),
]:
    auc = load_auc(p)
    if auc is not None:
        rows.append({"cell": "HiDF (in-dist)", "auc": auc, "kind": "in-dist"})
        break

# FF++ per manipulation
for m in ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]:
    auc = load_auc(os.path.join(eval_root, f"ffpp_cross_{m}", "metrics.json"))
    if auc is not None:
        rows.append({"cell": f"FF++ {m}", "auc": auc, "kind": "FF++ cross"})

# Celeb-DF
for p in [
    os.path.join(eval_root, "celebdf_test_metrics.json"),
    os.path.join(OUTPUT_DIR, "celebdf_test_metrics.json"),
]:
    auc = load_auc(p)
    if auc is not None:
        rows.append({"cell": "Celeb-DF v2", "auc": auc, "kind": "Celeb-DF cross"})
        break

if not rows:
    print("No AUC values found — eval cells haven't produced metrics.json files.")
else:
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUTPUT_DIR, "cross_dataset_summary.csv"), index=False)

    color_map = {"in-dist": "#2ecc71", "FF++ cross": "#3498db", "Celeb-DF cross": "#e67e22"}
    colors = [color_map[k] for k in df["kind"]]

    plt.figure(figsize=(11, 5.5))
    bars = plt.bar(df["cell"], df["auc"], color=colors, edgecolor="white", linewidth=1.2)
    plt.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="chance")
    plt.axhline(0.75, color="lightgray", linestyle=":", linewidth=1, label="gate 5 threshold (0.75)")
    plt.ylim(0, 1)
    plt.ylabel("AUC-ROC", fontsize=11)
    plt.title("EAHN (trained on HiDF) — cross-dataset AUC-ROC", fontsize=13, fontweight="bold")
    plt.xticks(rotation=30, ha="right")
    plt.legend(loc="upper right")
    for bar, val in zip(bars, df["auc"]):
        plt.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.015,
                 f"{val:.3f}", ha="center", fontsize=9)
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "cross_dataset_summary.png")
    plt.savefig(out, dpi=150)
    plt.show()
    print(f"\nSummary saved: {out}")
    print(f"CSV saved:     {os.path.join(OUTPUT_DIR, 'cross_dataset_summary.csv')}")
    print("\nTable:")
    print(df.to_string(index=False))


---
## Cell 11 — Heatmap overlays
Displays sample explanations (intrinsic, Grad-CAM, attention rollout) on HiDF test videos.


In [ ]:
import os, re
from collections import defaultdict
from IPython.display import display, HTML, Image

heatmap_dir = os.path.join(OUTPUT_DIR, "plots", "heatmaps")
if not os.path.isdir(heatmap_dir):
    # alternate location
    heatmap_dir = os.path.join(OUTPUT_DIR, "heatmaps")

if not os.path.isdir(heatmap_dir):
    print(f"No heatmaps directory found.")
    print("Check that scripts/HiDF_save_xai_overlays.py ran during Cell 7.")
else:
    methods = ["intrinsic", "gradcam", "rollout"]
    pngs = sorted(f for f in os.listdir(heatmap_dir) if f.endswith(".png"))

    if not pngs:
        print(f"No PNGs in {heatmap_dir}")
    else:
        pat = re.compile(r"(?P<vid>.+?)_(?P<lbl>real|fake)_conf(?P<p>[\d.]+)_(?P<method>\w+)_f(?P<f>\d+)\.png")
        grouped = defaultdict(lambda: defaultdict(list))
        for fname in pngs:
            m = pat.match(fname)
            if m:
                grouped[m.group("vid")][m.group("method")].append(fname)

        if not grouped:
            print("Could not parse heatmap filenames. First 5:")
            for f in pngs[:5]: print(f"   {f}")
        else:
            display(HTML(f"<h3>HiDF heatmap overlays — {len(grouped)} videos × {len(methods)} methods</h3>"))
            # Show 2 sample videos
            samples = sorted(grouped.keys())[:2]
            for vid in samples:
                display(HTML(f"<h4><code>{vid}</code></h4>"))
                for method in methods:
                    files = sorted(grouped[vid].get(method, []))
                    if not files: continue
                    display(HTML(f"<p><b>{method.upper()}</b> ({len(files)} frames)</p>"))
                    for fname in files[:6]:
                        display(Image(os.path.join(heatmap_dir, fname), width=200))

            display(HTML(f"<p style='color:#888'>Remaining {len(grouped)-len(samples)} "
                         f"videos in {heatmap_dir}/</p>"))


---
## Cell 12 — Build analysis-essentials bundle

A flat directory containing the key files for thesis review: every metrics.json,
every key plot, summary CSV, training history, one heatmap per method. No subdirectories,
no checkpoint weights, no face cache. Compact zip for fast review.


In [ ]:
import os, shutil, glob, json

bundle_dir = os.path.join(OUTPUT_DIR, "analysis_essentials")
if os.path.isdir(bundle_dir):
    shutil.rmtree(bundle_dir)
os.makedirs(bundle_dir, exist_ok=True)

eval_root = os.path.join(OUTPUT_DIR, "eval")
plot_root = os.path.join(OUTPUT_DIR, "plots")

copied = []
def grab(src, dst_name):
    """Copy src to bundle as dst_name if src exists."""
    if os.path.isfile(src):
        dst = os.path.join(bundle_dir, dst_name)
        shutil.copy2(src, dst)
        copied.append(dst_name)
        return True
    return False

# ── 1. All metrics JSON (renamed flat) ───────────────────────────────────────
grab(os.path.join(eval_root, "hidf_test_metrics.json"),    "01_hidf_metrics.json")
grab(os.path.join(eval_root, "metrics.json"),              "01_hidf_metrics.json")  # alt name
grab(os.path.join(OUTPUT_DIR, "metrics.json"),             "01_hidf_metrics.json")  # alt name

FFPP_METHODS = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]
for i, m in enumerate(FFPP_METHODS, start=2):
    grab(os.path.join(eval_root, f"ffpp_cross_{m}", "metrics.json"),
         f"0{i}_ffpp_{m}_metrics.json")

grab(os.path.join(eval_root, "celebdf_test_metrics.json"), "07_celebdf_metrics.json")
grab(os.path.join(OUTPUT_DIR, "celebdf_test_metrics.json"),"07_celebdf_metrics.json")

grab(os.path.join(OUTPUT_DIR, "explanation_metrics.json"), "08_explanation_metrics.json")

# ── 2. Summary tables ────────────────────────────────────────────────────────
grab(os.path.join(OUTPUT_DIR, "metrics.csv"),              "09_metrics_summary.csv")
grab(os.path.join(OUTPUT_DIR, "cross_dataset_summary.csv"),"10_cross_dataset_summary.csv")
grab(os.path.join(OUTPUT_DIR, "cross_dataset_summary.png"),"11_cross_dataset_summary.png")

# ── 3. Training curves ───────────────────────────────────────────────────────
grab(os.path.join(plot_root, "loss_curves.png"),           "12_loss_curves.png")
grab(os.path.join(plot_root, "val_accuracy_curves.png"),   "13_val_accuracy_curves.png")
grab(os.path.join(OUTPUT_DIR, "training_history.csv"),     "14_training_history.csv")
grab(os.path.join(OUTPUT_DIR, "logs.csv"),                 "14_training_history.csv")

# ── 4. Detection plots (ROC/PR/CM/score) per cell ────────────────────────────
def grab_cell_plots(prefix, idx):
    """Find ROC/PR/CM/score for one cell."""
    leaves = [
        ("roc_curve.png", "roc"),
        ("pr_curve.png", "pr"),
        ("confusion_matrix.png", "cm"),
        ("score_distribution.png", "score"),
    ]
    # Try plots/{prefix}_*.png first
    for leaf, short in leaves:
        candidates = [
            os.path.join(plot_root, f"{prefix}_{leaf}"),
            os.path.join(eval_root, prefix, leaf),
        ]
        # Also catch any file in eval/{prefix}/ that ends with leaf
        cell_dir = os.path.join(eval_root, prefix)
        if os.path.isdir(cell_dir):
            for fn in os.listdir(cell_dir):
                if fn.endswith(leaf):
                    candidates.append(os.path.join(cell_dir, fn))
        for c in candidates:
            if os.path.isfile(c):
                grab(c, f"{idx:02d}_{prefix}_{short}.png")
                break

grab_cell_plots("hidf",   20)
for i, m in enumerate(FFPP_METHODS, start=21):
    grab_cell_plots(f"ffpp_cross_{m}", i)
grab_cell_plots("celebdf", 26)

# ── 5. One heatmap per method (representative) ───────────────────────────────
heatmap_dir = os.path.join(OUTPUT_DIR, "plots", "heatmaps")
if not os.path.isdir(heatmap_dir):
    heatmap_dir = os.path.join(OUTPUT_DIR, "heatmaps")
if os.path.isdir(heatmap_dir):
    seen = set()
    for fname in sorted(os.listdir(heatmap_dir)):
        if not fname.endswith(".png"): continue
        for method in ["intrinsic", "gradcam", "rollout"]:
            if f"_{method}_" in fname and method not in seen:
                grab(os.path.join(heatmap_dir, fname), f"30_heatmap_{method}.png")
                seen.add(method)
                break
        if len(seen) == 3: break

# ── 6. README ────────────────────────────────────────────────────────────────
readme_lines = [
    "# EAHN-HiDF — Analysis Essentials",
    "",
    "Flat bundle of the key artefacts from the HiDF training + cross-dataset evaluation run.",
    "",
    "## Contents",
    "",
    "- 01_hidf_metrics.json — HiDF in-distribution test metrics (primary result)",
    "- 02–06_ffpp_*_metrics.json — FF++ per-manipulation cross-eval (100R+100F each)",
    "- 07_celebdf_metrics.json — Celeb-DF v2 cross-eval (100R+100F balanced)",
    "- 08_explanation_metrics.json — explanation suite output",
    "- 09_metrics_summary.csv — combined metric summary",
    "- 10_cross_dataset_summary.csv + 11_cross_dataset_summary.png — AUC across all 7 cells",
    "- 12_loss_curves.png, 13_val_accuracy_curves.png — training trajectory",
    "- 14_training_history.csv — per-epoch metrics",
    "- 20_hidf_*.png — HiDF in-distribution ROC / PR / confusion / score",
    "- 21–25_ffpp_cross_*_*.png — FF++ per-manipulation ROC / PR / confusion / score",
    "- 26_celebdf_*.png — Celeb-DF ROC / PR / confusion / score",
    "- 30_heatmap_*.png — representative explanation overlays (intrinsic / gradcam / rollout)",
    "",
    "For reproduction details and the full repo state, see the complete archive",
    "(`eahn_hidf_complete.zip`) and the project bible.",
]
with open(os.path.join(bundle_dir, "00_README.md"), "w") as f:
    f.write("\n".join(readme_lines))
copied.append("00_README.md")

# ── Report ───────────────────────────────────────────────────────────────────
print(f"Analysis bundle: {bundle_dir}")
print(f"Files included ({len(copied)}):")
for f in sorted(set(copied)):
    print(f"  {f}")


---
## Cell 13 — Build the two zip archives

Two zips produced in `/kaggle/working/`:

1. **`eahn_hidf_complete.zip`** — entire `OUTPUT_DIR`: checkpoint, face cache hash references, all plots, all heatmaps, all logs. Large (~200–500 MB depending on heatmap count).
2. **`eahn_hidf_analysis.zip`** — the flat `analysis_essentials/` bundle. Small (~5–15 MB). For thesis review and quick sharing.

Both download from Kaggle's **Output** tab after this cell runs.


In [ ]:
import shutil, os

# 1) Complete archive
zip_complete = "/kaggle/working/eahn_hidf_complete"
shutil.make_archive(zip_complete, "zip", OUTPUT_DIR)
size_complete = os.path.getsize(zip_complete + ".zip") / 1e6
print(f"Complete archive : {zip_complete}.zip   ({size_complete:.1f} MB)")

# 2) Analysis-essentials archive
bundle_dir = os.path.join(OUTPUT_DIR, "analysis_essentials")
if os.path.isdir(bundle_dir):
    zip_analysis = "/kaggle/working/eahn_hidf_analysis"
    shutil.make_archive(zip_analysis, "zip", bundle_dir)
    size_analysis = os.path.getsize(zip_analysis + ".zip") / 1e6
    n_files = sum(1 for _ in os.listdir(bundle_dir))
    print(f"Analysis archive : {zip_analysis}.zip   ({size_analysis:.1f} MB, {n_files} files flat)")
else:
    print("[WARN] analysis_essentials/ not found — Cell 12 didn't run or failed.")

print()
print("=" * 70)
print("Download from Kaggle Output tab:")
print("  eahn_hidf_complete.zip   (everything for reproduction)")
print("  eahn_hidf_analysis.zip   (flat bundle for review / thesis)")
print("=" * 70)
print()
print("If you need to keep the trained checkpoint for re-evaluation later:")
print("  1. Download eahn_hidf_complete.zip")
print("  2. Re-upload as a private Kaggle Dataset")
print("  3. Attach to a new notebook and run Cell 6c to restore")
